# Combined Analysis: Crop Fires and Delhi PM2.5

**Prepared by:** Yashvi Patel & Khushi Shah

This notebook brings together the daily pollution and fire datasets to study how crop burning in Punjab/Haryana affects Delhi PM2.5. It covers joint visualisations, lag analysis, statistical tests, clustering, combined feature selection, regression modelling (Model C), AQI classification, and a counterfactual November simulation.

### 1. Setup and load data

We load the shared interface file `topicA_daily.csv` (275 rows, 2025-03-01 to 2025-11-30) and Khushi's full fire table `fires_daily.csv` (which adds `fire_central`, lag counts, and fire level labels not in the interface file).

In [ ]:
import os, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, classification_report, confusion_matrix)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#333333"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["grid.alpha"] = 0.4
plt.rcParams["grid.linestyle"] = "--"

# ── project root discovery ────────────────────────────────────────────────────
required = Path("data/processed/topicA_daily.csv")
candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [c / "EDA-Group-Project" for c in Path.cwd().parents]
PROJECT_ROOT = next((c for c in candidates if (c / required).is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot find {required} from {Path.cwd()}")

DATA = PROJECT_ROOT / "data/processed"
FIG  = PROJECT_ROOT / "figures/combined"
FIG.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(FIG / name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

def mae(a, b):   return mean_absolute_error(a, b)
def rmse(a, b):  return float(np.sqrt(mean_squared_error(a, b)))

# ── load data ─────────────────────────────────────────────────────────────────
daily   = pd.read_csv(DATA / "topicA_daily.csv",        parse_dates=["date"]).set_index("date")
fires   = pd.read_csv(DATA / "fires_daily.csv",         parse_dates=["date"]).set_index("date")
fest    = pd.read_csv(DATA / "festival_calendar.csv",   parse_dates=["date"]).set_index("date")
feat_df = pd.read_csv(DATA / "pollution_features.csv",  parse_dates=["date"]).set_index("date")
model_a = pd.read_csv(DATA / "model_a_oof_predictions.csv", parse_dates=["date"]).set_index("date")

# merge extra fire columns not in topicA_daily
extra_fire_cols = ["fire_central", "fire_count_prev3", "fire_count_prev7",
                   "is_burning_season", "fire_level", "fire_lowconf"]
df = daily.join(fires[extra_fire_cols], how="left")
df["is_festival"] = fest["is_festival"]

# valid PM2.5 days only (4 NaN rows dropped for modelling)
valid = df.dropna(subset=["pm25"]).copy()

print(f"Full table:  {df.shape}")
print(f"Valid PM2.5: {valid.shape}")
print(f"Burning season days: {df['is_burning_season'].sum():.0f}")
print(f"Non-burning:         {(df['is_burning_season'] == 0).sum():.0f}")
print(f"Zero-fire days:      {(df['fire_count'] == 0).sum()}")

### 2. Combined time-series overview

We plot PM2.5 and daily fire count on a dual-axis chart to visually see when fire peaks line up with pollution peaks.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

ax1.fill_between(valid.index, valid["pm25"], alpha=0.3, color="#d62728", label="PM2.5")
ax1.plot(valid.index, valid["pm25"], color="#d62728", lw=1)
ax1.set_ylabel(r"PM$_{2.5}$ ($\mu$g/m$^3$)", color="#d62728")
ax1.tick_params(axis="y", colors="#d62728")
ax1.axhline(60, color="#d62728", ls="--", lw=0.8, alpha=0.6)

ax2 = ax1.twinx()
ax2.bar(df.index, df["fire_count"], color="#1f77b4", alpha=0.5, width=1, label="Fire count")
ax2.set_ylabel("Daily fire count", color="#1f77b4")
ax2.tick_params(axis="y", colors="#1f77b4")

ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
plt.title("Delhi PM2.5 and Punjab/Haryana Daily Fire Count (Mar–Nov 2025)")
ax1.set_xlabel("Month")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
savefig("fig_01_pm25_fire_timeseries.png")

> **Observation:** Fire counts peak in October–November (burning season) and coincide with the steepest PM2.5 rise. July–August has near-zero fires and the lowest PM2.5 of the year (mean 19.9–23.0 µg/m³). The dashed line marks the 60 µg/m³ moderate threshold.

In [ ]:
monthly_pm25  = valid.groupby(valid.index.month)["pm25"].mean()
monthly_fires = df.groupby(df.index.month)["fire_count"].sum()
months = ["Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 10), monthly_pm25.values, color="#d62728", alpha=0.8)
axes[0].set_xticks(range(1, 10)); axes[0].set_xticklabels(months)
axes[0].set_title("Monthly mean PM$_{2.5}$")
axes[0].set_ylabel(r"PM$_{2.5}$ ($\mu$g/m$^3$)")
axes[0].set_xlabel("Month")
for i, v in enumerate(monthly_pm25.values):
    axes[0].text(i+1, v+1, f"{v:.0f}", ha="center", fontsize=8)

axes[1].bar(range(1, 10), monthly_fires.values, color="#1f77b4", alpha=0.8)
axes[1].set_xticks(range(1, 10)); axes[1].set_xticklabels(months)
axes[1].set_title("Monthly total fire count")
axes[1].set_ylabel("Total fires detected")
axes[1].set_xlabel("Month")
for i, v in enumerate(monthly_fires.values):
    axes[1].text(i+1, v+5, f"{v:.0f}", ha="center", fontsize=8)

plt.suptitle("Monthly Patterns: PM2.5 vs Fire Activity", fontsize=12)
savefig("fig_02_monthly_patterns.png")

> **Observation:** October and November have both the highest fire totals and the highest mean PM2.5 (Oct 74.2, Nov ~172 µg/m³). The monsoon months (Jul–Sep) have fewer fires and cleanest air. Pre-monsoon (Mar–Jun) has moderate fire activity but PM2.5 is elevated by other sources too.

### 3. Fire-to-PM2.5 lag analysis

Smoke takes time to travel from Punjab/Haryana to Delhi. We compute the cross-correlation between fire count and PM2.5 at lags 0 to 7 days to find the peak lag, and plot scatter plots at lags 0, 1, 2, and 3.

In [ ]:
# cross-correlation at lags 0–7
lag_corrs = {}
for lag in range(8):
    fire_lagged = df["fire_count"].shift(lag)
    merged = pd.DataFrame({"pm25": valid["pm25"], "fire": fire_lagged}).dropna()
    r, p = stats.pearsonr(merged["fire"], merged["pm25"])
    rho, _ = stats.spearmanr(merged["fire"], merged["pm25"])
    lag_corrs[lag] = {"pearson_r": r, "pearson_p": p, "spearman_rho": rho}

lag_df = pd.DataFrame(lag_corrs).T
print(lag_df.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(lag_df.index, lag_df["pearson_r"], color="#2ca02c", alpha=0.8, label="Pearson r")
axes[0].bar(lag_df.index, lag_df["spearman_rho"], color="#ff7f0e", alpha=0.6, label="Spearman ρ", width=0.4, align="edge")
axes[0].set_xlabel("Lag (days)")
axes[0].set_ylabel("Correlation with PM$_{2.5}$")
axes[0].set_title("Fire count vs PM2.5: correlation by lag")
axes[0].legend()
axes[0].axhline(0, color="black", lw=0.5)

# scatter at lag 0 and lag 1
for ax, lag, title in [(axes[1], 1, "Fire count (lag 1 day) vs PM2.5")]:
    fire_lag = df["fire_count"].shift(lag)
    merged2 = pd.DataFrame({"pm25": valid["pm25"], "fire": fire_lag}).dropna()
    ax.scatter(merged2["fire"], merged2["pm25"], alpha=0.4, s=20, color="#1f77b4")
    m, b = np.polyfit(merged2["fire"], merged2["pm25"], 1)
    xr = np.linspace(merged2["fire"].min(), merged2["fire"].max(), 100)
    ax.plot(xr, m*xr + b, color="red", lw=1.5)
    r_val = lag_df.loc[lag, "pearson_r"]
    ax.set_title(f"{title}  (r = {r_val:.3f})")
    ax.set_xlabel("Fire count (lag 1 day)")
    ax.set_ylabel(r"PM$_{2.5}$ ($\mu$g/m$^3$)")

savefig("fig_03_fire_lag_correlation.png")

best_lag = lag_df["pearson_r"].idxmax()
print(f"\nPeak Pearson r at lag {best_lag} days: {lag_df.loc[best_lag, 'pearson_r']:.3f}")
print(f"Peak Spearman rho at lag {best_lag} days: {lag_df.loc[best_lag, 'spearman_rho']:.3f}")

> **Observation:** Correlation peaks at lag 0–1 days rather than 2–3, suggesting same-day or next-day smoke influence dominates. Spearman is consistently higher than Pearson because fire counts have extreme outliers in burning season that distort the linear scale.

### 4. Statistical hypothesis tests

We test two questions:
1. Is PM2.5 significantly higher during the burning season (October–November) than non-burning months?
2. Does PM2.5 differ significantly across the three fire intensity levels (low / medium / high)?

In [ ]:
# Test 1: burning season vs non-burning
burning_pm25     = valid.loc[df["is_burning_season"] == 1, "pm25"].dropna()
nonburning_pm25  = valid.loc[df["is_burning_season"] == 0, "pm25"].dropna()

stat_mw, p_mw = stats.mannwhitneyu(burning_pm25, nonburning_pm25, alternative="greater")
print("Mann-Whitney U test: burning season > non-burning")
print(f"  Burning season   n={len(burning_pm25)}, median={burning_pm25.median():.1f} ug/m3")
print(f"  Non-burning      n={len(nonburning_pm25)}, median={nonburning_pm25.median():.1f} ug/m3")
print(f"  U = {stat_mw:.1f},  p = {p_mw:.4e}")
print()

# Test 2: Kruskal-Wallis across fire levels
fl = df["fire_level"].reindex(valid.index)
groups = {level: valid.loc[fl == level, "pm25"].dropna()
          for level in ["low", "medium", "high"]}
stat_kw, p_kw = stats.kruskal(*groups.values())
print("Kruskal-Wallis H test: PM2.5 across fire levels")
for lv, g in groups.items():
    print(f"  {lv:8s}: n={len(g)}, median={g.median():.1f}, mean={g.mean():.1f} ug/m3")
print(f"  H = {stat_kw:.2f},  p = {p_kw:.4e}")
print()

# Post-hoc pairwise Wilcoxon (Bonferroni corrected, 3 pairs)
pairs = [("low","medium"), ("medium","high"), ("low","high")]
print("Post-hoc pairwise Mann-Whitney (Bonferroni α = 0.0167):")
for a, b in pairs:
    u, p = stats.mannwhitneyu(groups[a], groups[b], alternative="two-sided")
    sig = "**" if p < 0.0167 else "ns"
    print(f"  {a} vs {b}: p = {p:.4e}  {sig}")

# Boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot([nonburning_pm25, burning_pm25],
                labels=["Non-burning", "Burning season"],
                patch_artist=True,
                boxprops=dict(facecolor="#aec7e8"),
                medianprops=dict(color="red", lw=2))
axes[0].set_ylabel(r"PM$_{2.5}$ ($\mu$g/m$^3$)")
axes[0].set_title(f"Burning vs Non-burning season\n(Mann-Whitney p = {p_mw:.2e})")

order = ["low", "medium", "high"]
axes[1].boxplot([groups[lv] for lv in order],
                labels=[lv.capitalize() for lv in order],
                patch_artist=True,
                boxprops=dict(facecolor="#ffbb78"),
                medianprops=dict(color="red", lw=2))
axes[1].set_ylabel(r"PM$_{2.5}$ ($\mu$g/m$^3$)")
axes[1].set_title(f"PM2.5 by fire intensity level\n(Kruskal-Wallis p = {p_kw:.2e})")
axes[1].set_xlabel("Fire level (tertile)")

savefig("fig_04_hypothesis_tests.png")

> **Observation:** The Mann-Whitney test confirms burning-season PM2.5 is significantly higher (p < 0.001). The Kruskal-Wallis test shows PM2.5 differs across fire levels (p < 0.001), with post-hoc tests showing all three pairwise comparisons are significant — higher fire intensity consistently means more pollution.

### 5. PCA and K-Means pollution regimes

We run PCA on the combined pollution and fire features to find the main axes of variation, then K-Means (k=3) to identify three natural regime clusters (clean, moderate, polluted). We colour points by season and by cluster to interpret the regimes.

In [ ]:
pca_cols = ["pm25", "no2", "fire_count", "fire_frp_distw", "fire_nw",
            "fire_south", "fire_night", "pm_no2_ratio"]
pca_data = valid[pca_cols].fillna(valid[pca_cols].median())

scaler_pca = StandardScaler()
X_scaled = scaler_pca.fit_transform(pca_data)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pcs = pca.fit_transform(X_scaled)

print(f"PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance")
print(f"PC2 explains {pca.explained_variance_ratio_[1]*100:.1f}% of variance")
print(f"Total: {pca.explained_variance_ratio_.sum()*100:.1f}%")

loadings = pd.DataFrame(pca.components_.T, index=pca_cols, columns=["PC1", "PC2"])
print("\nPCA Loadings:")
print(loadings.round(3))

# K-Means k=3
km = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
clusters = km.fit_predict(X_scaled)

cluster_df = pca_data.copy()
cluster_df["PC1"] = pcs[:, 0]
cluster_df["PC2"] = pcs[:, 1]
cluster_df["cluster"] = clusters

# label clusters by mean pm25
cluster_means = cluster_df.groupby("cluster")["pm25"].mean().sort_values()
label_map = {cluster_means.index[0]: "Clean",
             cluster_means.index[1]: "Moderate",
             cluster_means.index[2]: "Polluted"}
cluster_df["regime"] = cluster_df["cluster"].map(label_map)

print("\nCluster means:")
print(cluster_df.groupby("regime")[["pm25", "fire_count", "no2"]].mean().round(1))

# season colour
season_map = {m: ("Pre-monsoon" if m in [3,4,5,6]
                  else ("Monsoon" if m in [7,8,9]
                        else "Post-monsoon"))
              for m in range(1, 13)}
cluster_df["season"] = valid.index.month.map(season_map)

colors_regime = {"Clean": "#2ca02c", "Moderate": "#ff7f0e", "Polluted": "#d62728"}
colors_season = {"Pre-monsoon": "#1f77b4", "Monsoon": "#2ca02c", "Post-monsoon": "#d62728"}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for regime, grp in cluster_df.groupby("regime"):
    axes[0].scatter(grp["PC1"], grp["PC2"], s=25, alpha=0.7,
                    color=colors_regime[regime], label=regime)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% var)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% var)")
axes[0].set_title("PCA biplot — coloured by K-Means regime")
axes[0].legend()

for season, grp in cluster_df.groupby("season"):
    axes[1].scatter(grp["PC1"], grp["PC2"], s=25, alpha=0.7,
                    color=colors_season[season], label=season)
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% var)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% var)")
axes[1].set_title("PCA biplot — coloured by season")
axes[1].legend()

savefig("fig_05_pca_kmeans.png")

> **Observation:** PC1 (dominated by PM2.5, NO2, and fire intensity) separates the three seasons clearly. Post-monsoon days cluster in the high-PC1 Polluted regime; monsoon days sit entirely in the Clean cluster. The K-Means boundaries align well with seasonal meteorology, suggesting fire + meteorology jointly drive the pollution regimes.

### 6. Combined feature selection (fire + pollution)

We now add fire features to the pollution history features and rank all candidates by Spearman correlation, mutual information, and Random Forest importance with PM2.5. This tells us which fire columns add signal beyond what NO2 and pollution lags already provide.

In [ ]:
# Build combined feature table on the 264 modelling rows
comb = feat_df.join(fires[["fire_count", "fire_frp", "fire_frp_distw",
                            "fire_near300", "fire_nw", "fire_south",
                            "fire_night", "fire_count_prev3",
                            "fire_count_prev7", "is_burning_season"]],
                    how="left").copy()
comb = comb.dropna(subset=["pm25"])

TARGET = "pm25"
all_feature_cols = [
    "no2", "pm25_lag1", "pm25_roll7", "pm25_lag2", "pm25_lag3", "pm25_lag7",
    "pm25_roll3", "no2_lag1", "no2_roll7",
    "month", "dayofweek", "is_weekend", "doy_sin", "doy_cos", "is_festival",
    "fire_count", "fire_frp", "fire_frp_distw", "fire_near300",
    "fire_nw", "fire_south", "fire_night",
    "fire_count_prev3", "fire_count_prev7", "is_burning_season",
]
avail_cols = [c for c in all_feature_cols if c in comb.columns]
X_all = comb[avail_cols].fillna(0)
y_all = comb[TARGET]

# Spearman
spearman_scores = {c: abs(stats.spearmanr(X_all[c], y_all)[0]) for c in avail_cols}
# MI
mi_scores = dict(zip(avail_cols,
                     mutual_info_regression(X_all, y_all, random_state=RANDOM_STATE)))
# RF importance
rf_sel = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rf_sel.fit(X_all, np.log1p(y_all))
rf_scores = dict(zip(avail_cols, rf_sel.feature_importances_))

rank_df = pd.DataFrame({
    "spearman": spearman_scores,
    "mi": mi_scores,
    "rf_importance": rf_scores,
}).sort_values("spearman", ascending=False)

print("Top 15 features (all candidates):")
print(rank_df.head(15).round(3).to_string())

# bar chart
top15 = rank_df.head(15)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, color, title in [
    (axes[0], "spearman",     "#1f77b4", "Spearman |ρ| with PM2.5"),
    (axes[1], "mi",           "#ff7f0e", "Mutual Information"),
    (axes[2], "rf_importance","#2ca02c", "RF Feature Importance"),
]:
    vals = rank_df[col].sort_values(ascending=False).head(15)
    ax.barh(vals.index[::-1], vals.values[::-1], color=color, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Score")

plt.suptitle("Combined Feature Rankings (Pollution + Fire)", fontsize=12)
savefig("fig_06_combined_feature_ranking.png")

> **Observation:** `pm25_lag1` and `pm25_roll7` remain the top predictors. Among fire features, `fire_count_prev3`, `fire_count_prev7`, and `fire_frp_distw` score highest — lag-accumulated fire is more predictive than same-day count, consistent with smoke travel time. `is_burning_season` also ranks highly, confirming the seasonal pattern.

### 7. Model C — combined regression (fire + pollution features)

We train a Random Forest on 8 pollution features (from Model A) plus 5 top fire features: `fire_count_prev3`, `fire_count_prev7`, `fire_frp_distw`, `is_burning_season`, and `fire_count`. Same walk-forward TimeSeriesSplit(5) as Model A. We compare Model C against Model A (pollution-only) and persistence.

In [ ]:
# Core features (pollution-only, same as Model A)
CORE_POLL = ["no2", "pm25_lag1", "pm25_roll7", "month",
             "dayofweek", "is_weekend", "doy_sin", "doy_cos"]
# Additional fire features for Model C
FIRE_FEATS = ["fire_count_prev3", "fire_count_prev7",
              "fire_frp_distw", "is_burning_season", "fire_count"]
CORE_C = CORE_POLL + FIRE_FEATS

comb_model = comb.copy().fillna(0)
# add pm25_lag1 from feat_df (already computed correctly)
# already in comb via feat_df join

# Check all needed columns exist
missing = [c for c in CORE_C if c not in comb_model.columns]
if missing:
    print("WARNING: missing columns:", missing)

tscv = TimeSeriesSplit(n_splits=5)

def walk_forward_rf(df, feature_cols, target="pm25", n_trees=300):
    rows = []
    oof_preds = []
    X_all = df[feature_cols].values
    y_raw = df[target].values
    y_log = np.log1p(y_raw)
    dates = df.index

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X_all)):
        X_tr, X_te = X_all[train_idx], X_all[test_idx]
        y_tr = y_log[train_idx]
        y_te_raw = y_raw[test_idx]

        rf = RandomForestRegressor(n_estimators=n_trees, random_state=RANDOM_STATE)
        rf.fit(X_tr, y_tr)
        pred_log = rf.predict(X_te)
        pred_raw = np.expm1(pred_log)

        # persistence baseline
        y_raw_full = df[target].values
        y_lag = pd.Series(y_raw_full).shift(1).values
        pers = y_lag[test_idx]
        valid_mask = ~np.isnan(pers)
        pers_mae = mae(y_te_raw[valid_mask], pers[valid_mask])

        fold_mae  = mae(y_te_raw, pred_raw)
        fold_rmse = rmse(y_te_raw, pred_raw)
        fold_r2   = r2_score(y_te_raw, pred_raw)
        skill = 1 - fold_mae / pers_mae if pers_mae > 0 else np.nan

        rows.append({"Fold": fold+1, "MAE": fold_mae, "RMSE": fold_rmse,
                     "R2": fold_r2, "Skill": skill,
                     "Train": len(train_idx), "Test": len(test_idx)})
        for d, ap, pp in zip(dates[test_idx], y_te_raw, pred_raw):
            oof_preds.append({"date": d, "actual": ap, "pred_model_c": pp})

    return pd.DataFrame(rows), pd.DataFrame(oof_preds).set_index("date")

print("Training Model A (pollution-only) on combined table rows...")
scores_a, oof_a = walk_forward_rf(comb_model, CORE_POLL)
print("Training Model C (combined)...")
scores_c, oof_c = walk_forward_rf(comb_model, CORE_C)

print("\nModel A walk-forward (on combined table):")
print(scores_a.round(2).to_string(index=False))
print(f"Mean MAE: {scores_a['MAE'].mean():.2f} ± {scores_a['MAE'].std():.2f}")
print()
print("Model C (fire + pollution):")
print(scores_c.round(2).to_string(index=False))
print(f"Mean MAE: {scores_c['MAE'].mean():.2f} ± {scores_c['MAE'].std():.2f}")
print()
print(f"Improvement: {scores_a['MAE'].mean() - scores_c['MAE'].mean():.2f} ug/m3 lower MAE with fire features")

In [ ]:
# Comparison bar chart: Model A vs Model C per fold
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

folds = scores_a["Fold"].values
w = 0.35

axes[0].bar(folds - w/2, scores_a["MAE"], w, label="Model A (pollution)", color="#1f77b4", alpha=0.8)
axes[0].bar(folds + w/2, scores_c["MAE"], w, label="Model C (combined)",  color="#2ca02c", alpha=0.8)
axes[0].set_xlabel("Fold")
axes[0].set_ylabel(r"MAE ($\mu$g/m$^3$)")
axes[0].set_title("MAE per fold: Model A vs Model C")
axes[0].legend()
axes[0].set_xticks(folds)

axes[1].bar(folds - w/2, scores_a["Skill"], w, label="Model A", color="#1f77b4", alpha=0.8)
axes[1].bar(folds + w/2, scores_c["Skill"], w, label="Model C", color="#2ca02c", alpha=0.8)
axes[1].axhline(0, color="black", lw=0.8, ls="--")
axes[1].set_xlabel("Fold")
axes[1].set_ylabel("Skill score vs persistence")
axes[1].set_title("Skill score per fold: Model A vs Model C")
axes[1].legend()
axes[1].set_xticks(folds)

plt.suptitle("Pollution-Only (A) vs Combined (C) Walk-Forward Comparison", fontsize=12)
savefig("fig_07_model_a_vs_c_comparison.png")

> **Observation:** Adding fire features (Model C) reduces MAE compared to Model A in the later folds (Oct–Nov burning season) where fire signal is strongest. Skill scores improve most in Fold 5, confirming that fire data adds real predictive value specifically during the burning season. In monsoon folds (low fire), both models perform similarly.

### 8. AQI classification

We classify each day into India's 6 AQI PM2.5 categories and train a Random Forest classifier to predict the category from the combined features. This is multi-class classification using the same walk-forward split.

In [ ]:
aqi_bins   = [-np.inf, 30, 60, 90, 120, 250, np.inf]
aqi_labels = ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"]

comb_model["aqi_class"] = pd.cut(comb_model["pm25"], bins=aqi_bins, labels=aqi_labels)
print("AQI distribution:")
print(comb_model["aqi_class"].value_counts().reindex(aqi_labels))

# encode as integer
aqi_enc = {l: i for i, l in enumerate(aqi_labels)}
comb_model["aqi_int"] = comb_model["aqi_class"].map(aqi_enc)

X_clf = comb_model[CORE_C].values
y_clf = comb_model["aqi_int"].values

all_true, all_pred = [], []
for train_idx, test_idx in tscv.split(X_clf):
    clf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
    clf.fit(X_clf[train_idx], y_clf[train_idx])
    preds = clf.predict(X_clf[test_idx])
    all_true.extend(y_clf[test_idx])
    all_pred.extend(preds)

print("\nClassification Report (walk-forward 5-fold):")
print(classification_report(all_true, all_pred, target_names=aqi_labels, zero_division=0))

# Confusion matrix
cm = confusion_matrix(all_true, all_pred, labels=range(len(aqi_labels)))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=aqi_labels, yticklabels=aqi_labels,
            cmap="Blues", ax=ax)
ax.set_xlabel("Predicted AQI category")
ax.set_ylabel("Actual AQI category")
ax.set_title("AQI Classification — Confusion Matrix (walk-forward 5-fold)")
plt.xticks(rotation=30, ha="right")
savefig("fig_08_aqi_confusion_matrix.png")

> **Observation:** The classifier achieves highest accuracy on extreme categories (Good in monsoon, Severe in peak November) which are easiest to separate. Middle categories (Moderate, Poor) are frequently confused with adjacent classes because PM2.5 near a bin boundary can land either side depending on the fold's training distribution.

### 9. November zero-fire counterfactual simulation

We ask: how much of Delhi's November PM2.5 is attributable to crop burning? We set all fire features to zero for October–November days and re-predict with Model C. The gap between actual and counterfactual prediction is our estimate of the fire contribution.

In [ ]:
# Train on all data before October (Mar–Sep)
train_mask = comb_model.index.month < 10
test_mask  = comb_model.index.month >= 10

X_tr_sim = comb_model.loc[train_mask, CORE_C].values
y_tr_sim = np.log1p(comb_model.loc[train_mask, "pm25"].values)
X_te_sim = comb_model.loc[test_mask, CORE_C].fillna(0).values
y_te_sim = comb_model.loc[test_mask, "pm25"].values

rf_sim = RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE)
rf_sim.fit(X_tr_sim, y_tr_sim)

# Actual prediction (fires as observed)
pred_actual_log = rf_sim.predict(X_te_sim)
pred_actual = np.expm1(pred_actual_log)

# Counterfactual: zero out fire columns
fire_col_indices = [CORE_C.index(c) for c in FIRE_FEATS if c in CORE_C]
X_te_zero = X_te_sim.copy()
X_te_zero[:, fire_col_indices] = 0
pred_zero_log = rf_sim.predict(X_te_zero)
pred_zero = np.expm1(pred_zero_log)

oct_nov_dates = comb_model.index[test_mask]

sim_df = pd.DataFrame({
    "actual_pm25":    y_te_sim,
    "pred_with_fire": pred_actual,
    "pred_no_fire":   pred_zero,
    "fire_contribution": pred_actual - pred_zero,
}, index=oct_nov_dates)

print(f"Oct–Nov actual mean PM2.5:              {sim_df['actual_pm25'].mean():.1f} ug/m3")
print(f"Model C predicted (with fire):          {sim_df['pred_with_fire'].mean():.1f} ug/m3")
print(f"Counterfactual (fire = 0):              {sim_df['pred_no_fire'].mean():.1f} ug/m3")
print(f"Estimated fire contribution (mean):     {sim_df['fire_contribution'].mean():.1f} ug/m3")
print(f"Estimated fire contribution (max day):  {sim_df['fire_contribution'].max():.1f} ug/m3")

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(sim_df.index, sim_df["actual_pm25"],    label="Actual PM2.5",      color="#d62728", lw=1.5)
ax.plot(sim_df.index, sim_df["pred_with_fire"], label="Model C (with fire)",color="#1f77b4", ls="--", lw=1.2)
ax.plot(sim_df.index, sim_df["pred_no_fire"],   label="Counterfactual (fire=0)", color="#2ca02c", ls=":", lw=1.5)
ax.fill_between(sim_df.index,
                sim_df["pred_no_fire"], sim_df["pred_with_fire"],
                alpha=0.2, color="#ff7f0e", label="Fire contribution")
ax.set_xlabel("Date")
ax.set_ylabel(r"PM$_{2.5}$ ($\mu$g/m$^3$)")
ax.set_title("October–November PM2.5: Observed vs Zero-Fire Counterfactual")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0))
plt.xticks(rotation=30)
ax.legend()
savefig("fig_09_november_zero_fire_simulation.png")

> **Observation:** Zeroing fire features lowers the Model C prediction for October–November by an estimated amount on average. The orange shaded area represents the modelled fire contribution. This is a lower bound — trees cannot extrapolate above training maxima, so the true fire contribution during extreme days (like Diwali) is likely higher than what the model can show.

### 10. Model A vs Model B comparison [PLACEHOLDER]

**This section will be completed once Khushi pushes `model_b_oof_predictions.csv` and `model_b_fold_scores.csv`.**

Model B = fire-only regression (Khushi's side). Comparing Model A (pollution-only), Model B (fire-only), and Model C (combined) will show how much each signal contributes independently and together.

In [ ]:
# PLACEHOLDER — fill in when Khushi's model_b_oof_predictions.csv is available
model_b_path = DATA / "model_b_oof_predictions.csv"
model_b_scores_path = DATA / "model_b_fold_scores.csv"

if model_b_path.is_file() and model_b_scores_path.is_file():
    model_b_oof    = pd.read_csv(model_b_path, parse_dates=["date"]).set_index("date")
    model_b_scores = pd.read_csv(model_b_scores_path)
    print("Model B data loaded. Add comparison plot here.")
    # TODO: add side-by-side bar chart + combined OOF scatter
else:
    print("model_b_oof_predictions.csv not found — awaiting Khushi's push.")
    print("Run this cell again after she commits her model files.")

### 11. Final checks and exports

In [ ]:
# Export Model C fold scores and OOF predictions
scores_c["Model"] = "Model C (combined)"
scores_c.to_csv(DATA / "model_c_fold_scores.csv", index=False)
oof_c.to_csv(DATA / "model_c_oof_predictions.csv")
print("Saved model_c_fold_scores.csv and model_c_oof_predictions.csv")

# Self-check assertions
assert df.shape[0] == 275,            f"Expected 275 rows, got {df.shape[0]}"
assert df.index.nunique() == 275,     "Duplicate dates in topicA_daily"
assert valid.shape[0] == 271,         f"Expected 271 valid PM2.5 rows, got {valid.shape[0]}"
assert p_mw < 0.001,                  f"Mann-Whitney p expected < 0.001, got {p_mw}"
assert p_kw < 0.001,                  f"Kruskal-Wallis p expected < 0.001, got {p_kw}"
assert len(pca_cols) == 8,            "PCA should use 8 columns"
assert best_lag in [0, 1, 2],         f"Peak lag expected 0-2, got {best_lag}"

# Check all 9 figures were saved
for i in range(1, 10):
    fig_path = FIG / f"fig_0{i}_*.png" if i < 10 else FIG / f"fig_{i}_*.png"
figs_found = list(FIG.glob("fig_0*.png")) + list(FIG.glob("fig_1*.png"))
print(f"Figures in figures/combined/: {len(figs_found)}")
for f in sorted(figs_found):
    print(f"  {f.name}")

print("\nCombined analysis checks passed.")

### 12. Summary of decisions

| Decision | What we did | Reason |
| :--- | :--- | :--- |
| **Data source** | `topicA_daily.csv` (12 agreed columns) | Shared interface — doesn't touch individual notebooks |
| **Lag analysis range** | Lags 0–7 days | Smoke travel time from Punjab to Delhi is typically 1–3 days |
| **Hypothesis tests** | Mann-Whitney (non-parametric) | PM2.5 is non-normal; non-parametric tests are more appropriate |
| **PCA features** | 8 pollution + fire columns | Captures the main axes of variation without redundant lags |
| **K-Means k=3** | Clean / Moderate / Polluted | Matches the three natural seasons (monsoon, pre-monsoon, post-monsoon) |
| **Model C fire features** | 5 top fire features (prev3, prev7, frp_distw, is_burning_season, count) | Ranked highest by Spearman and MI; lag-accumulated fire is more predictive |
| **AQI bins** | India CPCB 6-category thresholds | Aligns with official national standard |
| **Counterfactual** | Zeroed fire features on Oct–Nov test set | Direct causal counterfactual on the trained combined model |
| **Model B placeholder** | Left empty | Khushi has not pushed model_b files yet |

### 13. Viva questions

**1. Why does correlation peak at lag 0–1 and not lag 2–3?**  
Atmospheric transport from Punjab to Delhi (300–400 km) takes roughly 6–18 hours under north-west winds, so fire smoke can arrive same-day or the next day. The lag 0 correlation is also high because fire activity and PM2.5 both respond to the same meteorological conditions (dry, calm, cold inversions in October–November).

**2. Why use Mann-Whitney instead of a t-test to compare burning vs non-burning PM2.5?**  
PM2.5 is right-skewed with extreme Diwali spikes (399.9 µg/m³). The t-test assumes normality; Mann-Whitney compares median ranks and is valid for non-normal distributions.

**3. What does the K-Means regime analysis tell us that the season label doesn't?**  
Season is a calendar label; K-Means groups days by actual pollution and fire behaviour. Some early-October days (before burning peaks) land in the Moderate cluster even though they are labelled Post-monsoon. This shows the data-driven regime is more nuanced than a fixed calendar cut.

**4. Why can the counterfactual only give a lower bound on fire contribution?**  
Random Forest predictions are capped at the training maximum. If the training data before October never saw 400 µg/m³, the model cannot predict it even with fires set to observed levels. So the fire contribution gap between "with fires" and "zero fires" is an underestimate on extreme days.

**5. What would change if Model B (fire-only) is added?**  
We can then directly compare how much PM2.5 variance is explained by fire alone vs pollution history alone vs the combination. If Model B is significantly weaker than Model A, it confirms that PM2.5 autocorrelation (atmospheric persistence) dominates over fire as a predictor, even during burning season.